# NeuroSLM — Bio-Inspired Language Model (~258M)

Neuroscience-inspired LM with genome-encoded algorithms, differential attention, mixture-of-depths, NT-modulated attention, predictive coding, Hebbian traces, and epigenetic self-optimization.

**Training resumes automatically across runtime disconnects.**
Checkpoints + memory + evolved DNA are persisted to Git LFS.

| Preset | Params | GPU | VRAM | Batch |
|--------|--------|-----|------|-------|
| `large` | ~100M | T4 | 15GB | 4 |
| **`xl`** | **~258M** | **A100** | **40GB** | **4** |

**Novel mechanisms (no existing model has all of these):**
1. Differential attention (dual-softmax noise cancellation)
2. Mixture-of-Depths (dynamic per-token layer skipping)
3. NT-modulated attention temperature (per-head, state-dependent)
4. Predictive coding inter-layer loss (deep supervision)
5. Hebbian fast-weight attention traces (in-context learning accelerator)
6. Genome-encoded module algorithms (evolvable via epigenetic optimizer)

**Setup:** Runtime → Change runtime type → **A100** → Set `GITHUB_TOKEN` in cell 2

In [ ]:
# 1) GPU + torch check
import subprocess, sys

print(f'Python {sys.version}')
!nvidia-smi

# Check torch CUDA (subprocess to avoid poisoning kernel)
r = subprocess.run(
    [sys.executable, "-c",
     "import torch; print(torch.__version__); print(torch.cuda.is_available())"],
    capture_output=True, text=True
)
lines = r.stdout.strip().split('\n')
torch_ver = lines[0] if lines else '?'
has_cuda = len(lines) > 1 and lines[1] == 'True'
print(f'\ntorch {torch_ver}, CUDA={has_cuda}')

if not has_cuda:
    print('\n' + '='*60)
    print('❌ torch has NO CUDA support!')
    print('')
    print('This usually means Colab gave you a CPU-only runtime.')
    print('FIX: Runtime → Disconnect and delete runtime')
    print('     Runtime → Change runtime type → A100 (or T4)')
    print('     Then re-run all cells from the top.')
    print('')
    print('If nvidia-smi above shows a GPU, torch is mismatched.')
    print('In that case, restart runtime and re-run — do NOT')
    print('pip install anything before running this notebook.')
    print('='*60)
else:
    import torch
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {name} ({vram:.1f} GB)')
    if vram < 35:
        print('⚠️ <40GB VRAM — use preset="large" (170M) instead of "xl"')
    else:
        print('✓ Ready for 3B model')

In [ ]:
# 2) Clone repo with PAT for push access
# ─────────────────────────────────────────────────────────────────────────────
# Token priority:
#   1. Hardcoded value below (leave blank for security)
#   2. Colab Secret named  GITHUB  (Settings -> Secrets -> Add new secret)
#   3. Colab Secret named  GITHUB_TOKEN  (legacy name)
# ─────────────────────────────────────────────────────────────────────────────
GITHUB_TOKEN = ""  # paste PAT here if not using Colab Secrets

import os, re, subprocess

try:
    from google.colab import userdata as _ud
    GITHUB_TOKEN = GITHUB_TOKEN or _ud.get('GITHUB') or _ud.get('GITHUB_TOKEN') or ''
except Exception:
    pass

GITHUB_TOKEN = GITHUB_TOKEN.strip()

if GITHUB_TOKEN:
    os.environ['GITHUB'] = GITHUB_TOKEN
    print("Token ready")
else:
    print("No token - checkpoints will not persist to Git")

# Build authenticated URL (token is credential, never echoed)
_base = 'https://github.com/269652/neuroslm.git'
REPO  = f'https://{GITHUB_TOKEN}@github.com/269652/neuroslm.git' if GITHUB_TOKEN else _base

if os.path.exists('/content/neuroslm/.git'):
    # Already cloned (runtime reconnect) - fix the remote URL and pull latest
    print("Repo already present - updating remote URL and pulling latest...")
    subprocess.run(['git', 'remote', 'set-url', 'origin', REPO],
                   cwd='/content/neuroslm', check=False)
    subprocess.run(['git', 'pull', '--ff-only'],
                   cwd='/content/neuroslm', check=False)
    os.chdir('/content/neuroslm')
else:
    # Fresh clone
    os.chdir('/content')
    get_ipython().system(f'git clone {REPO}')
    os.chdir('/content/neuroslm')

# Ensure git identity is set (required for commits inside Colab)
subprocess.run(['git', 'config', 'user.email', 'train@neuroslm'], check=False)
subprocess.run(['git', 'config', 'user.name',  'NeuroSLM Train'],  check=False)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# 3) Install deps + Git LFS + restore checkpoints + DNA
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA! Go to Runtime → Disconnect and delete runtime → "
        "Change runtime type → A100/T4 → re-run from cell 1"
    )
print(f"✓ torch {torch.__version__} CUDA GPU={torch.cuda.get_device_name(0)}")

# Install deps (torch is already CUDA from Colab — don't touch it)
!pip install -q numpy tiktoken datasets tqdm einops networkx
!pip install -q --no-deps accelerate

# Git LFS
!apt-get install -y git-lfs -qq 2>/dev/null
!git lfs install
!git lfs pull

# Restore LFS checkpoints + memory + evolved DNA
import glob, shutil, os
os.makedirs('/content/checkpoints', exist_ok=True)
for ext in ['*.pt', '*.mem', '*.mem.json', '*.dna.json']:
    for c in sorted(glob.glob(f'lfs_checkpoints/{ext}')):
        dest = os.path.join('/content/checkpoints', os.path.basename(c))
        if not os.path.exists(dest):
            shutil.copy2(c, dest)
            print(f"  Restored: {os.path.basename(c)}")

CKPT_DIR = locals().get('CKPT_DIR', '/content/checkpoints')
existing = sorted(glob.glob(CKPT_DIR + '/*.pt'))
dna_files = sorted(glob.glob('checkpoints/*.dna.json'))
if existing:
    print(f"\n✓ {len(existing)} checkpoint(s). Latest: {existing[-1]}")
    ckpt = torch.load(existing[-1], map_location='cpu', weights_only=False)
    print(f"  Step {ckpt.get('step', '?')}")
    has_dna = 'module_genomes' in ckpt or 'epigenetic' in ckpt
    print(f"  Evolved DNA in checkpoint: {has_dna}")
    if dna_files:
        print(f"  DNA snapshots: {len(dna_files)} (.dna.json files)")
    del ckpt
else:
    print("No checkpoints — training from scratch")

In [ ]:
# 4) Configuration — change these to control training
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

PRESET = "xl"            # "large" (100M, T4) | "xl" (350M, A100) | "xxl" (10B, multi-GPU)
TOTAL_STEPS = 100_000    # Target: 100K steps × batch_size × ctx = ~800M tokens
BATCH_SIZE = 1           # use grad_accum for effective larger batch
GRAD_ACCUM = 4           # effective batch = BATCH_SIZE * GRAD_ACCUM = 4
SAVE_EVERY = 500         # Checkpoint + push frequency
CKPT_DIR = '/content/checkpoints'  # absolute path — works after reconnects
MODE = "mix"             # "mix" = interleave text + chat (best for instruction following)
CHAT_RATIO = 0.6         # 60% chat/instruction, 40% text/knowledge

# Auto-detect: if GPU < 40GB VRAM, fall back to large preset
import torch
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram < 35 and PRESET == "xl":
        print(f"⚠️ Only {vram:.0f}GB VRAM — falling back to large preset (100M)")
        PRESET = "large"
        BATCH_SIZE = 1
    else:
        print(f"✓ {vram:.0f}GB VRAM — using {PRESET} preset (batch={BATCH_SIZE})")

print(f"\nTraining plan:")
print(f"  Preset:     {PRESET}")
print(f"  Steps:      {TOTAL_STEPS:,}")
print(f"  Batch:      {BATCH_SIZE} x {GRAD_ACCUM} grad_accum = {BATCH_SIZE*GRAD_ACCUM} effective")
print(f"  Mode:       {MODE} (chat_ratio={CHAT_RATIO})")
print(f"  Save every: {SAVE_EVERY}")

In [ ]:
# 5) ABLATION: Baseline vs Full Model (1000 steps each)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Trains both models for 1000 steps to confirm bio modules improve over a
# param-matched vanilla transformer BEFORE committing to the full 100K run.
import os
if "GITHUB" not in os.environ:
    try:
        from google.colab import userdata as _ud
        _tok = _ud.get("GITHUB") or _ud.get("GITHUB_TOKEN") or ""
        if _tok: os.environ["GITHUB"] = _tok
    except Exception: pass

os.environ["PYTHONUNBUFFERED"] = "1"

print("=" * 60)
print("  ABLATION: Baseline vs Full Model (1000 steps each)")
print("=" * 60)

# --- Baseline: param-matched vanilla transformer ---
print("\n▶ Training BASELINE (param-matched vanilla transformer)...")
_bl_cmd = (f"cd /content/neuroslm && python -u -m neuroslm.train"
           f" --preset {PRESET}"
           f" --steps 1000"
           f" --batch_size {BATCH_SIZE}"
           f" --grad_accum {GRAD_ACCUM}"
           f" --ckpt_dir /content/checkpoints_baseline"
           f" --device cuda"
           f" --mode {MODE}"
           f" --chat_ratio {CHAT_RATIO}"
           f" --save_every 1000"
           f" --log_every 50"
           f" --seed 42"
           f" --baseline")
print(_bl_cmd)
!{_bl_cmd}

# --- Full model: all bio modules ---
print("\n▶ Training FULL MODEL (bio modules)...")
_fm_cmd = (f"cd /content/neuroslm && python -u -m neuroslm.train"
           f" --preset {PRESET}"
           f" --steps 1000"
           f" --batch_size {BATCH_SIZE}"
           f" --grad_accum {GRAD_ACCUM}"
           f" --ckpt_dir /content/checkpoints_full"
           f" --device cuda"
           f" --mode {MODE}"
           f" --chat_ratio {CHAT_RATIO}"
           f" --save_every 1000"
           f" --log_every 50"
           f" --seed 42")
print(_fm_cmd)
!{_fm_cmd}

print("\n✓ Both ablation runs complete. Run next cell to compare.")

In [ ]:
# 5b) Compare ablation results
import glob, torch

print("=" * 60)
print("  ABLATION RESULTS: 1000 steps")
print("=" * 60)

for label, ckpt_dir in [
    ("BASELINE (param-matched vanilla)", "/content/checkpoints_baseline"),
    ("FULL MODEL  (bio modules)",        "/content/checkpoints_full"),
]:
    ckpts = sorted(glob.glob(f'{ckpt_dir}/*.pt'))
    if ckpts:
        ckpt = torch.load(ckpts[-1], map_location='cpu', weights_only=False)
        step = ckpt.get('step', '?')
        cfg_d = ckpt.get('cfg', {})
        n_params = sum(v.numel() for v in ckpt['model'].values())
        is_baseline = cfg_d.get('baseline', False)
        print(f"\n  {label}:")
        print(f"    Params:     {n_params/1e6:.2f}M")
        print(f"    Step:       {step}")
        print(f"    Baseline:   {is_baseline}")
        del ckpt
    else:
        print(f"\n  {label}: No checkpoint found in {ckpt_dir}/")

print("\n" + "=" * 60)
print("  Check step-by-step loss in the training output above.")
print("  FULL MODEL loss < BASELINE → bio modules help → run cell 6.")
print("  FULL MODEL loss ≥ BASELINE → tune loss weights first.")
print("=" * 60)

In [ ]:
# 6) Full training — RESUMES AUTOMATICALLY from latest checkpoint
# Run this after the ablation (cells 5 + 5b) confirms bio modules help.
import os, sys
if "GITHUB" not in os.environ:
    try:
        from google.colab import userdata as _ud
        _tok = _ud.get("GITHUB") or _ud.get("GITHUB_TOKEN") or ""
        if _tok: os.environ["GITHUB"] = _tok
    except Exception: pass

os.environ["PYTHONUNBUFFERED"] = "1"

_cmd = (f"cd /content/neuroslm && python -u -m neuroslm.train"
        f" --preset {PRESET}"
        f" --steps {TOTAL_STEPS}"
        f" --batch_size {BATCH_SIZE}"
        f" --grad_accum {GRAD_ACCUM}"
        f" --ckpt_dir {CKPT_DIR}"
        f" --device cuda"
        f" --meta"
        f" --mode {MODE}"
        f" --chat_ratio {CHAT_RATIO}"
        f" --save_every {SAVE_EVERY}"
        f" --resume latest")
print(_cmd)
!{_cmd}

In [ ]:
# 7) Push checkpoint + memory + DNA to Git LFS
import glob, shutil, os, subprocess, shlex

ckpts = sorted(glob.glob(CKPT_DIR + '/*.pt'))
mems  = sorted(glob.glob(CKPT_DIR + '/*.mem'))
dnas  = sorted(glob.glob(CKPT_DIR + '/*.dna.json'))

if not ckpts:
    print('No checkpoints to push. Train first (cell 6).')
else:
    os.makedirs('lfs_checkpoints', exist_ok=True)
    pushed = []
    for f in ckpts[-2:] + mems[-2:] + dnas[-2:]:
        dest = os.path.join('lfs_checkpoints', os.path.basename(f))
        shutil.copy2(f, dest)
        pushed.append(os.path.basename(f))
    print(f"Prepared: {pushed}")

    token = os.environ.get('GITHUB', '')
    if token:
        remote = f"https://{token}@github.com/269652/neuroslm.git"
        subprocess.run(['git', 'remote', 'set-url', 'origin', remote], check=False)

    subprocess.run(f"git add -f {shlex.quote('lfs_checkpoints')}/*", shell=True)
    subprocess.run(['git', 'commit', '-m', f"chkpt: {pushed[-1] if pushed else 'update'}"], check=False)
    r = subprocess.run(['git', 'push'], check=False)
    print('✓ Pushed' if r.returncode == 0 else '✖ Push failed — check token')

In [ ]:
# 7) Benchmarks — HellaSwag, ARC-Easy, ARC-Challenge, MMLU
# Compares against SmolLM2, TinyLlama, Phi-3, Qwen2.5, Llama-3.1
import glob
ckpts = sorted(glob.glob(CKPT_DIR + '/*.pt'))
if ckpts:
    latest = ckpts[-1]
    print(f"Benchmarking: {latest}")
    !python -m neuroslm.benchmarks --ckpt {latest} --preset {PRESET} --device cuda --max_samples 500
else:
    print("No checkpoints. Train first.")

In [ ]:
# 8) Generate text
import glob
ckpts = sorted(glob.glob(CKPT_DIR + '/*.pt'))
if ckpts:
    latest = ckpts[-1]
    print(f'Using: {latest}')
    !python -m neuroslm.generate --ckpt {latest} --preset {PRESET} --prompt "Explain the theory of relativity in simple terms" --max_new 512 --temperature 0.7 --top_k 40 --device cuda
else:
    print('No checkpoints. Train first.')

In [ ]:
# 9) Intelligence Metrics — consciousness, identity drift, causal density
import glob, torch, sys
sys.path.insert(0, '.')
from neuroslm.config import PRESETS
from neuroslm.brain import Brain
from neuroslm.tokenizer import Tokenizer

ckpts = sorted(glob.glob(CKPT_DIR + '/*.pt'))
assert ckpts, "No checkpoints. Train first."
latest = ckpts[-1]

tok = Tokenizer()
cfg = PRESETS[PRESET]()
cfg.vocab_size = tok.vocab_size
brain = Brain(cfg).to("cuda")
ckpt = torch.load(latest, map_location="cuda", weights_only=False)
brain.load_state_dict(ckpt["model"], strict=False)
brain.eval()
n_params = sum(p.numel() for p in brain.parameters())
print(f"Model: {n_params/1e6:.1f}M params, step {ckpt.get('step', '?')}")

# Load memory if available
import os
mem_path = latest.replace('.pt', '.mem')
if os.path.exists(mem_path):
    brain.load_memory_checkpoint(mem_path)
    print(f"Memory loaded: {mem_path}")

# Show intelligence metrics
brain.metrics.observe_narrative(brain.narrative_system)
brain.metrics.observe_memory(brain.episodic, brain.consolidated, brain.causal)
snap = brain.metrics.snapshot()
print("\n" + "=" * 60)
print("  INTELLIGENCE & CONSCIOUSNESS METRICS")
print("=" * 60)
for k, v in snap.items():
    if isinstance(v, float):
        print(f"  {k:30s}: {v:.4f}")
    else:
        print(f"  {k:30s}: {v}")
print("=" * 60)
print(f"\n  Causal rules learned: {brain.causal.stats()}")
print(f"  Memory gate stats: {brain.comprehension_gate.stats()}")

In [ ]:
# 10) Inspect Module Algorithms — Genome alleles → Opcodes → Lisp → Module Params
# The genome IS the program. Alleles directly encode opcode logits + operands.
# Pipeline: ModuleGenome.alleles → decompile → Lisp → extract params → push to modules
import glob, torch, sys, os
sys.path.insert(0, '.')
from neuroslm.config import PRESETS
from neuroslm.brain import Brain
from neuroslm.tokenizer import Tokenizer

ckpts = sorted(glob.glob(CKPT_DIR + '/*.pt'))
assert ckpts, "No checkpoints. Train first."
latest = ckpts[-1]

tok = Tokenizer()
cfg = PRESETS[PRESET]()
cfg.vocab_size = tok.vocab_size
brain = Brain(cfg).to("cpu")
ckpt = torch.load(latest, map_location="cpu", weights_only=False)
brain.load_state_dict(ckpt["model"], strict=False)

# Restore module genomes if saved
if 'module_genomes' in ckpt and hasattr(brain, 'module_genomes'):
    from neuroslm.dna.compiler import ModuleGenomePool
    brain.module_genomes = ModuleGenomePool.from_state(ckpt['module_genomes'])
    brain._recompile_all_genomes()
    print(f"✓ Restored module genomes from step {ckpt.get('step', '?')}")

brain.eval()
print(f"Loaded step {ckpt.get('step', '?')}\n")

# Show compilation report
if hasattr(brain, 'genome_compiler'):
    print(brain.genome_compilation_report())

    # Show genome summaries — what each module's algorithm looks like
    print("\n" + "=" * 60)
    print("  MODULE GENOME ALGORITHMS (genome IS the program)")
    print("=" * 60)
    for region, genome in brain.module_genomes.active_all().items():
        print(f"\n  {genome.summary()}")
        print(f"    Generation: {genome.generation}, Fitness: {genome.fitness:.4f}")
        # Show extracted params that actually control the module
        env = brain.genome_compiler.get_env(region)
        param_keys = [k for k in env if not k.startswith('_') and k not in
                      ('__region__', 'layers', 'connections', 'learning_rule',
                       'projections', 'nt_production')]
        if param_keys:
            params_str = ", ".join(f"{k}={env[k]:.3f}" for k in sorted(param_keys)
                                   if isinstance(env.get(k), (int, float)))
            print(f"    Params → module: {params_str}")

    print("\n" + "=" * 60)
    print("  COMPILED LISP (decompiled from genome alleles)")
    print("=" * 60)
    for name, lisp_src in brain.get_all_module_lisp().items():
        print(f"\n{'─' * 60}")
        print(lisp_src)

    # Save to files
    out_dir = brain.genome_compiler.save_all_lisp('compiled_programs')
    print(f"\n✓ Saved compiled .lisp files to {out_dir}/")
    for f in sorted(os.listdir(out_dir)):
        print(f"  {f}")
else:
    print("No genome compiler found (baseline model)")

## Multi-day training workflow

1. **First run:** Run cells 1–6 sequentially. Training starts from scratch.
2. **Runtime disconnects:** Colab disconnects after ~12h. Checkpoints are auto-saved + pushed.
3. **Resume:** Re-run cells 1–5. Cell 5 auto-detects the latest checkpoint and continues.
4. **Track progress:** Run cell 7 (benchmarks) periodically to see improvement vs SOTA.
5. **Ablation:** Run cell 6b instead of 5 to compare baseline vs full model (critical first step).
6. **Target:** 100K steps ≈ 800M tokens at batch=4, ctx=2048.

### Training data pipeline
- **Text:** FineWeb-Edu (10B tokens curated web), Cosmopedia, TinyStories
- **Chat:** OpenHermes-2.5, UltraChat-200k, WildChat-1M, SlimOrca, hh-rlhf, Dolly
- **Mode `mix`:** 60% chat + 40% text (tunable via `CHAT_RATIO`)

### What gets persisted across restarts
- `.pt` checkpoint (model weights + optimizer + genome state + epigenetic marks)
- `.mem` memory checkpoint (episodic + consolidated + causal)
- `.dna.json` evolved DNA snapshot (inspectable JSON)

### Benchmark targets (258M xl preset)
| Benchmark | Random | Target | SOTA ~300M (SmolLM2) |
|-----------|--------|--------|---------------------|
| HellaSwag | 0.25 | >0.35 | 0.42 |
| ARC-Easy | 0.25 | >0.40 | 0.48 |
| ARC-Challenge | 0.25 | >0.28 | 0.30 |
| MMLU | 0.25 | >0.28 | 0.30 |

In [ ]:
# Cell: Commit & push LFS checkpoints from notebook
import os
import subprocess

# Configure repo remote with token if available
token = os.environ.get("GITHUB_TOKEN", "")
GITHUB_USER = os.environ.get("GITHUB_USER", "269652")
REPO_NAME = os.environ.get("GITHUB_REPO", "neuroslm")
if token:
    remote_url = f"https://{token}@github.com/{GITHUB_USER}/{REPO_NAME}.git"
    print("Setting remote URL with token (will not echo token)...")
    subprocess.run(["git", "remote", "set-url", "origin", remote_url], check=False)

# Ensure folder exists
path = os.path.join("neuroslm", "lfs_checkpoints")
if not os.path.exists(path):
    print(f"No '{path}' folder found. Nothing to push.")
else:
    # Force-add (in case .gitignore would skip LFS files), commit, and push
    print("Adding files to git (force add)")
    subprocess.run(["git", "add", "-f", os.path.join(path, "*")], shell=True)
    print("Committing")
    subprocess.run(["git", "commit", "-m", "Add LFS checkpoints from notebook"], check=False)
    print("Pushing to origin")
    res = subprocess.run(["git", "push"], check=False)
    if res.returncode == 0:
        print("Push succeeded")
    else:
        print("Push failed — check token/remote/permissions")


In [ ]:
# Cell: Commit & push LFS checkpoints from notebook (explicit)
import os, subprocess, shlex

lfs_dir = os.path.join('neuroslm', 'lfs_checkpoints')
if not os.path.isdir(lfs_dir):
    print(f"Folder not found: {lfs_dir} — nothing to push")
else:
    token = os.environ.get('GITHUB_TOKEN', '')
    gh_user = os.environ.get('GITHUB_USER', '269652')
    gh_repo = os.environ.get('GITHUB_REPO', 'neuroslm')

    if token:
        remote = f"https://{token}@github.com/{gh_user}/{gh_repo}.git"
        print("Setting git remote with token (token not printed)")
        subprocess.run(['git','remote','set-url','origin', remote], check=False)
    else:
        print('No GITHUB_TOKEN found in environment — using existing remote')

    # Force-add files (in case .gitignore blocks them)
    cmd_add = f"git add -f {shlex.quote(lfs_dir)}/*"
    print('Running:', cmd_add)
    subprocess.run(cmd_add, shell=True)

    # Commit
    commit_msg = 'Add LFS checkpoints from notebook'
    subprocess.run(['git','commit','-m', commit_msg], check=False)

    # Push
    print('Pushing to origin...')
    r = subprocess.run(['git','push'], check=False)
    if r.returncode == 0:
        print('Push succeeded')
    else:
        print('Push failed — check token and remote URL')
